In [1]:
# imports

# show rich formats 
from IPython.display import display, Markdown

# get python to interact with openai
from openai import OpenAI

# use personal openai key
import os
from dotenv import load_dotenv

# check load_dotenv works - should come back 'True'
# load_dotenv()

# make a destination 'resumes' directory for the work

os.makedirs("resumes", exist_ok=True)

# use python to format markdown to html
from markdown import markdown

#### because we're sending this out as a resume, we want it to be in a .pdf format. 
#### that means we'll have to use a library called weasy... 
##### <i><b>< record scratch ></i></b>
#### nope. sorry. 
#### not using weasyprint. lining weasyprint libraries up with each particular python environmet and setting / dedicating kernals still causes nightmares of 'public-speaking-in-underpants' proportions. surely a powerful tool, but it's got the playfulness and ease of use of a rabid porcupine. <i>no grazie</n>.
#### instead, going with pdfkit. working in html, so it'll do the job.
##### * note: pdfkit works using wkhtmltopdf, which <i>in very simple terms</i> converts a webpage to a pdf file. please see [reference.txt](https://github.com/npj210mlk/jobapp_prompter/blob/main/requirements.txt) in the repo for installation instructions.


In [2]:
# import pdfkit

# # test
# pdfkit.from_string("<h1>It should be called 'QueezyPrint,' amirite?</h1>", "output.pdf")

#### ha! apparently, jupyter agrees - came back 'True'

***
#### with the libraries imported we can now break the project down into four (4) steps:
##### 1.) open and read the markdown resume file
#####     * see requirements notes
##### 2.) input the desired job description
##### 3.) template some prompt engineering
##### 4.) convert markdown to html
##### 5.) convert html to pdf
***
*** 
#### Step 1: Open and Read the Markdown Resume
****

In [3]:
# open resume and read it
with open("/Users/nicholasjoseph/Desktop/nj_master_resume.md", "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

# check to see if it worked:
# display(Markdown(resume_string))

# markdown resume displays as expected.

***
#### Step 2: Input the Desired Job Description
***

In [4]:
# job description will be from anywhere, so input is used to pause the program until we find one to copy/paste into this variable 
job_desc_string = input()

 About Dobot  Founded in 2015, Dobot Robotics is the creator of the world's first desktop grade collaborative robot. We offer 6 main product lines: CR, CRS, MG400, M1 Pro, Nova, and Magician, with more than a dozen of collaborative robot models.  We are the first in the industry to offer a product line up that covers 0.5 to 20 kg payloads. To date, Dobot has sold over 68,000 collaborative robots to 140 countries and regions and has ranked first in Chinese robot exporter by volume 4 consecutive years.  Dobot robots are currently operational across over 15 industries including consumer electronics, automotive, metal processing, semiconductor, healthcare, chemical and retail.    Why Join Us  At Dobot, our people are humble, intelligent, compassionate and creative. We create to liberate all extraneous labor and realize optimal human-machine collaboration. We dream to make real change to the world. Join us to make this happen with a career at Dobot.    About the Team  Located in Carrollton 

***
#### Step 3: Template Some Prompt Engineering
##### - this involves dealing with ChatGPT, particularly the ChatGPT 4o-mini pre-trained transformer.
##### <u>a couple of things about the ChatGPT 4o-mini engine (model)</u>:
#####     a.) it is the smaller, more affordable version of the massive GPT-4o engine available to developers; and
#####     b.) because its focus is on text classificaton / prediction, it is purely a "decoder-only" type transformer
#### The idea is to have ChatGPT create the prompt to respond to the likely Applicant Tracking System being used by the job poster.
##### Reason being, machines talk to machines better than we can. 
***

In [5]:
# making up a random "lambda" function to create the variable "prompt_template"

# the text is what I would be putting into ChatGPT were I to do this one job at a time - prompt engineering

prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts
    - Select only the core competencies most relevant to the job description

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)

3. **Summary Selection**:
    - Please tailor the best best Summary format for the job description and Recruiter expectations.

4. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

5. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.

6. **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

In [6]:
# set the prompt variable for our ChatGPT message roles
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 4: Generate the Resume with GPT-4o-mini
##### - Make the api call and tell GPT to write the resume using the prompt from above
***

In [7]:
# set up api client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

# make the call

# set response variable to hold the all info we get back
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    # set our roles up - think of casting a play: "Today, the role of the Expert Resume Writer will be played by the system."
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
    
    # give it some creative license 
    temperature = 0.7
)

# get our response
response_string = response.choices[0].message.content

***
#### Step 5: Show Us the Results
***

In [8]:
# separate new resume from improvements AI suggests at 'Additional Suggestions'
response_list = response_string.split("## Additional Suggestions")

In [9]:
display(Markdown(response_list[0]))

# Nicholas Joseph
**Product Manager | Servant Leader | Tech Enthusiast**

Email: [nickpjoseph210@gmail.com](mailto:nickpjoseph210@gmail.com) | Phone: (210) 771-9853 | LinkedIn: [nickjoseph](https://www.linkedin.com/in/nickjoseph) | [GitHub](https://github.com/npj210mlk)

---

### Career Summary  

Dynamic Product Manager with over 10 years of experience driving data-driven solutions and managing cross-functional teams within the tech and robotics sectors. Expertise in developing and maintaining relationships with stakeholders to enhance product offerings and drive sales performance. Proven ability to identify market opportunities and deliver innovative solutions that ensure customer satisfaction and business growth.

---

### Relevant Experience

**Cloud Data Engineer**  
**Data Clymer (now Spaulding Ridge)**, Remote | 07/2022 - 06/2023  
- Spearheaded the design and implementation of data pipelines for major sports leagues, achieving a 98.9% reduction in data collection times, significantly enhancing operational efficiency and user experience.  
- Collaboratively facilitated Agile ceremonies and delivered training sessions to enhance team performance and ensure alignment with project goals.  
- Enhanced reporting accuracy by 53% through a successful database upgrade, directly impacting decision-making capabilities.

**Data Engineer**  
**Gathi Analytics (now Apexon)**, Remote | 12/2020 - 03/2022  
- Transformed complex banking data into cloud architecture, improving accessibility and performance, and ensuring compliance with regulatory standards.  
- Developed interactive dashboards based on data analysis, empowering decision-makers with real-time insights for strategic planning.  
- Actively mentored new hires, fostering a culture of continuous learning and knowledge sharing within the organization.

**Project Manager**  
**Noble Hands H & C**, San Antonio, TX | 10/2017 - 01/2020 | 07/2023 - Present  
- Managed multiple projects with budgets under $1M, achieving zero overages and a 5-star customer rating through effective stakeholder engagement and risk management.  
- Developed and executed strategic plans to enhance project delivery, achieving high customer satisfaction and repeat business.  
- Cultivated strong relationships with cross-functional teams, driving collaboration and performance enhancement.

---

### Skills  

- **Sales & Business Development**: Lead Generation, Customer Relationship Management, Sales Strategy, Market Intelligence, Stakeholder Engagement  
- **Technical Proficiency**: SQL, Python, Data Analysis and Visualization, Cloud Computing, Data Transformation Tools, API Integration  
- **Project Management**: Agile Methodologies (Scrum), Risk Management, Change Management, Team Leadership, KPI Development  
- **Soft Skills**: Interpersonal Communication, Presentation Skills, Problem-solving, Mentoring, Cross-functional Collaboration  

---

### Education  

**Certificate of Completion** - PMP Exam Prep - 35 PDU  
Technical Institute of America, Remote | 2024  

**Certificate of Completion** - Data Science  
Codeup, San Antonio, TX | 2020  

**Bachelor's** - Biology  
Concordia University, Austin, TX | 2001  

---



***
#### Step 6: Save the New Resume
##### Great. Hit several snags. You either need WeasyPrint installed in some capacity, or other tools I found (like 'Grip') are difficult to automate and test on MacOS. 
##### Looks like the programming gods have spoken: we're going with WeasyPrint.
##### <center><span style ="color:red"><b><u>To Do This:</u></b></span><center>
##### 1.) completely uninstall existing WeasyPrint's existence from your machine with pip and brew uninstalls;
##### 2.) run a 'brew cleanup' just to wipe out any remnants;
##### 3.) form home directory in Terminal (I'm using Mac), type 'brew install cairo pango gdk-pixbuf libffi' - these are the native Weasyprint dependencies;
##### 4.) set your environment variables in your profile (for me, my ~/.zshrc file) to the following:
###### export DYLD_LIBRARY_PATH=/opt/homebrew/lib:$DYLD_LIBRARY_PATH
###### export LIBRARY_PATH="/opt/homebrew/lib:$LIBRARY_PATH"
###### export PKG_CONFIG_PATH="/opt/homebrew/lib/pkgconfig:/opt/homebrew/share/pkgconfig"
###### export PATH="/opt/homebrew/bin:$PATH"
##### 5.) restart the terminal;
##### 6.) navigate to your project folder;
##### 7.) type 'pip install weasyprint markdown'; and (finally)
##### 8.) open your notebook from your project file with 'jupyter notebook'
***

In [10]:
# Markdown was already imported earlier
# import WeasyPrint and its HTML abilities
from weasyprint import HTML

# save it as PDF
output_pdf_file = "resumes/nick_joseph_resume.pdf"

# convert the Markdown file to HTML
html_content = markdown(response_list[0])

# now take that HTML and convert it into a PDF and save
HTML(string=html_content).write_pdf(output_pdf_file)

In [11]:
# let's save the new Markdown file, too
markdown_output = "resumes/nickjoseph_chatgpt_resume.md"

with open(markdown_output, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

***
#### Step 7: Display Suggestions for Improvement
##### Because we split "response_string" earlier, it was split at "Additional Suggestions," giving two items "response_list."
##### The first item ([0]) is the resume ChatGPT wrote with our prompt.
##### The second item ([1]) are the suggestions ChatGPT offers to make our resume stronger
***

In [12]:
# show us how to make the resume stronger
display(Markdown(response_list[1]))



1. **Skills to Add**:  
   - Familiarity with robotics and automation technologies, specifically collaborative robotic systems.  
   - Proficiency in sales forecasting and reporting tools.

2. **Certifications or Courses**:  
   - Consider pursuing certifications in Robotics or Automation (e.g., Certified Automation Professional).  
   - Courses in Sales Strategy and Techniques for Technical Products could enhance your sales acumen.

3. **Project Ideas or Experiences**:  
   - Engage in projects that involve the integration of robotic solutions in manufacturing or industrial environments.  
   - Seek opportunities to collaborate with system integrators or attend trade shows related to robotics to gain industry insights and expand your network.  

These enhancements will help align your profile more closely with the requirements of the Dobot role, emphasizing your ability to connect technical solutions with business needs.

In [ ]:
# # markdown is already imported
# from weasyprint import HTML

In [ ]:
# from markdown2pdf import convert

# markdown_resume = display(Markdown(response_list[0]))

# convert(markdown_resume, "nj_resume_in_pdf.pdf")